# RAG 4手法比較ノートブック

`baseline` / `chunk_optimizer` / `reranker` / `hybrid_search` の評価結果を可視化します。

## 前提

1. 各スクリプトを実行して `results/` に CSV が出力されている前提：
   ```bash
   python -m src.generate_data
   python -m src.baseline           # 検索のみ評価（Ollama不要）
   python -m src.chunk_optimizer    # 5パターン比較
   python -m src.reranker           # CrossEncoder リランキング
   python -m src.hybrid_search      # Vector+BM25 ハイブリッド
   ```
2. このノートブックを起動：
   ```bash
   jupyter notebook notebooks/comparison.ipynb
   ```

結果がない場合は、本ノートブック末尾に**サンプル可視化（モックデータ）**セルを用意していますので、そちらで挙動を確認できます。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 日本語フォント
plt.rcParams['font.family'] = ['Yu Gothic', 'MS Gothic', 'Meiryo', 'Hiragino Sans']
plt.rcParams['axes.unicode_minus'] = False

RESULTS = Path('../results')
print(f'結果ディレクトリ: {RESULTS.resolve()}')
print(list(RESULTS.glob('*.csv')))

## 1. 手法別 Recall@3 / MRR の取得

各手法の最新CSVから `# Recall@3` `# MRR` `# Mean_elapsed_sec` のサマリーを抽出します。

In [ ]:
def parse_summary(csv_path):
    """先頭の # 行からサマリーを取得"""
    summary = {}
    with open(csv_path, encoding='utf-8') as f:
        for line in f:
            if not line.startswith('#'):
                break
            parts = [p.strip() for p in line.lstrip('#').strip().split(',')]
            if len(parts) >= 2:
                summary[parts[0]] = parts[1]
    return summary

rows = []
for method in ['baseline', 'reranker', 'hybrid_search']:
    files = sorted(RESULTS.glob(f'{method}_*.csv'))
    if files:
        s = parse_summary(files[-1])
        rows.append({
            'method': method,
            'recall@3': float(s.get('Recall@3', 0)),
            'mrr': float(s.get('MRR', 0)),
            'elapsed': float(s.get('Mean_elapsed_sec', 0)),
        })

if rows:
    df = pd.DataFrame(rows)
    print(df)
else:
    print('CSV未生成。先に各 src/*.py を実行してください。')
    df = None

## 2. 手法別 Recall@3 棒グラフ

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(df['method'], df['recall@3'], color=['#6c757d', '#1a3a5c', '#2e7d32'])
    ax.set_ylabel('Recall@3')
    ax.set_title('手法別 Recall@3')
    ax.set_ylim(0, 1.0)
    for i, v in enumerate(df['recall@3']):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
    plt.tight_layout()
    plt.show()

## 3. 手法別 MRR 棒グラフ

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(df['method'], df['mrr'], color=['#6c757d', '#1a3a5c', '#2e7d32'])
    ax.set_ylabel('MRR')
    ax.set_title('手法別 MRR (Mean Reciprocal Rank)')
    ax.set_ylim(0, 1.0)
    for i, v in enumerate(df['mrr']):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
    plt.tight_layout()
    plt.show()

## 4. チャンクサイズ別 精度の折れ線

In [ ]:
chunk_csv = RESULTS / 'chunk_optimizer_summary.csv'
if chunk_csv.exists():
    cdf = pd.read_csv(chunk_csv)
    cdf = cdf.sort_values('chunk_size')
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(cdf['method'], cdf['recall_at_k'], marker='o', label='Recall@3', color='#1a3a5c')
    ax.plot(cdf['method'], cdf['mrr'], marker='s', label='MRR', color='#c62828')
    ax.set_ylabel('精度')
    ax.set_title('チャンクサイズ × オーバーラップ別 精度')
    ax.set_xticklabels(cdf['method'], rotation=20, ha='right')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('chunk_optimizer_summary.csv が見つかりません。`python -m src.chunk_optimizer` を先に実行してください。')

## 5. 応答時間と精度のトレードオフ散布図

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(df['elapsed'], df['recall@3'], s=200, c=['#6c757d', '#1a3a5c', '#2e7d32'])
    for _, r in df.iterrows():
        ax.annotate(r['method'], (r['elapsed'], r['recall@3']), xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel('平均応答時間 (秒)')
    ax.set_ylabel('Recall@3')
    ax.set_title('応答時間 vs 精度（左上ほど望ましい）')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---

## 6. 結果がない時の挙動確認用：モックデータ可視化

実機実行前にノートブックの動作確認をしたい場合、以下のセルでサンプルグラフが描けます。

In [ ]:
# モックデータ（実行結果が出るまでの仮表示）
mock_df = pd.DataFrame([
    {'method': 'baseline',      'recall@3': 0.55, 'mrr': 0.42, 'elapsed': 0.12},
    {'method': 'reranker',      'recall@3': 0.72, 'mrr': 0.61, 'elapsed': 0.34},
    {'method': 'hybrid_search', 'recall@3': 0.78, 'mrr': 0.65, 'elapsed': 0.18},
])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(mock_df['method'], mock_df['recall@3'], color=['#6c757d', '#1a3a5c', '#2e7d32'])
axes[0].set_title('[Mock] Recall@3 比較')
axes[0].set_ylim(0, 1.0)
axes[1].bar(mock_df['method'], mock_df['mrr'], color=['#6c757d', '#1a3a5c', '#2e7d32'])
axes[1].set_title('[Mock] MRR 比較')
axes[1].set_ylim(0, 1.0)
plt.tight_layout()
plt.show()
print('※ これはモック値です。実機実行後の実数値ではありません。')